In [13]:
import click
import tempfile
import queue
import sys
import threading
import sounddevice as sd
import soundfile as sf
import numpy

assert numpy  # avoid "imported but unused" message (W0611)


In [14]:

class AudioRecorder:
    recorders = {}  # Holds active recorder threads

    def __init__(self, filename=None, device=None, samplerate=None, channels=1, subtype=None):
        self.filename = filename
        self.device = device
        self.samplerate = samplerate
        self.channels = channels
        self.subtype = subtype
        self.q = queue.Queue()
        self.thread = None
        self.stop_event = threading.Event()

    def int_or_str(self, text):
        """Helper function for input conversion."""
        try:
            return int(text)
        except ValueError:
            return text

    def list_devices(self):
        """List audio devices."""
        print(sd.query_devices())

    def callback(self, indata, frames, time, status):
        """This is called (from a separate thread) for each audio block."""
        if status:
            print(status, file=sys.stderr)
        self.q.put(indata.copy())

    def setup_samplerate(self):
        """Determine the samplerate if not provided."""
        if self.samplerate is None:
            device_info = sd.query_devices(self.device, 'input')
            self.samplerate = int(device_info['default_samplerate'])

    def _record(self):
        """Internal method to handle recording in a separate thread."""
        try:
            self.setup_samplerate()

            if self.filename is None:
                self.filename = tempfile.mktemp(prefix='delme_rec_unlimited_',
                                                suffix='.wav', dir='')

            # Make sure the file is opened before recording anything
            with sf.SoundFile(self.filename, mode='x', samplerate=self.samplerate,
                              channels=self.channels, subtype=self.subtype) as file:
                with sd.InputStream(samplerate=self.samplerate, device=self.device,
                                    channels=self.channels, callback=self.callback):
                    print('#' * 80)
                    print('Recording... Press Ctrl+C or stop via command.')
                    print('#' * 80)
                    while not self.stop_event.is_set():
                        file.write(self.q.get())

        except Exception as e:
            print(f"Error occurred: {type(e).__name__}: {str(e)}")
            sys.exit(1)

    def start_recording(self):
        """Start the recording in a separate thread."""
        if self.thread is None or not self.thread.is_alive():
            self.stop_event.clear()
            self.thread = threading.Thread(target=self._record)
            self.thread.start()
            AudioRecorder.recorders[self.filename] = self.thread
            print(f"Recording started: {self.filename}")
        else:
            print("Recording is already running.")

    def stop_recording(self):
        """Stop the recording thread."""
        if self.thread and self.thread.is_alive():
            self.stop_event.set()
            self.thread.join()
            del AudioRecorder.recorders[self.filename]
            print(f"Recording stopped: {self.filename}")
        else:
            print("No active recording to stop.")


    @classmethod
    def list_recordings(cls):
        """List all active recording threads."""
        if cls.recorders:
            print("Active recordings:")
            for filename, thread in cls.recorders.items():
                print(f"- {filename}: {'Running' if thread.is_alive() else 'Stopped'}")
        else:
            print("No active recordings.")



In [15]:
rec = AudioRecorder()
# rec.list_devices()
rec.__init__(filename='test.wav', device=1, samplerate=44100, channels=1, subtype='PCM_24')


In [16]:

rec.start_recording()


Recording started: test.wav


################################################################################
Recording... Press Ctrl+C or stop via command.
################################################################################


In [18]:
rec.stop_recording()

Recording stopped: test.wav
